## גרפים של משתנה המטרה לפי שעת ואורך מסלול

In [ ]:
# הגדרת קבוצות
short_routes = train_df[train_df['route_length_km'] <= train_df['route_length_km'].quantile(0.25)]
mid_routes = train_df[
    (train_df['route_length_km'] > train_df['route_length_km'].quantile(0.25)) & 
    (train_df['route_length_km'] <= train_df['route_length_km'].quantile(0.75))
]
long_routes = train_df[train_df['route_length_km'] > train_df['route_length_km'].quantile(0.75)]

print(f"Short routes (< {train_df['route_length_km'].quantile(0.25):.1f} km): {len(short_routes):,}")
print(f"Mid routes:                                                          {len(mid_routes):,}")
print(f"Long routes (> {train_df['route_length_km'].quantile(0.75):.1f} km): {len(long_routes):,}")

fig, axes = plt.subplots(3, 1, figsize=(18, 18))

for ax, df_group, title in zip(axes, 
                                [short_routes, mid_routes, long_routes],
                                ['Short Routes', 'Mid Routes', 'Long Routes']):
    pivot = pd.crosstab(df_group['full_hour'], df_group['target'], normalize='index') * 100
    pivot[['early', 'on_time', 'delay']].plot(
        kind='bar', stacked=True, ax=ax,
        color=['steelblue', 'green', 'red'],
        edgecolor='white'
    )
    
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', label_type='center',
                     fontsize=7, color='white', fontweight='bold')
    
    ax.set_title(f'Target Distribution by Hour - {title}', fontsize=14)
    ax.set_xlabel('Hour', fontsize=12)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(['Early', 'On Time', 'Delay'], loc='upper right')

plt.tight_layout()
plt.show()

## New Features

### time_of_day

In [1]:
def get_time_of_day(hour):
    if hour <= 5:   return 'night'
    elif hour <= 9: return 'morning'
    elif hour <= 14: return 'midday'
    elif hour <= 18: return 'afternoon'
    else:           return 'evening'

### Route Length Category


In [ ]:
# מסלול קצר/בינוני/ארוך
df['route_length_cat'] = pd.qcut(df['route_length_km'], q=3, labels=['short', 'mid', 'long'])

### Is Night

In [ ]:
# לילה = סיכוי גבוה להקדמה
df['is_night'] = df['full_hour'].isin([0,1,2,3,4,5]).astype(int)

### Interaction: Night + Long Route

In [ ]:
# לילה + מסלול ארוך = סיכוי גבוה מאוד להקדמה
df['night_x_long_route'] = df['is_night'] * df['route_length_km']

### Historical Early Rate per Route and hour

In [ ]:
# Historical Early Rate per Route + Hour
early_rate = train_df.groupby(['route_id', 'full_hour'])['target'].apply(
    lambda x: (x == 'early').mean()
).reset_index(name='route_hour_early_rate')

# החל על כל הסטים
train_df = train_df.merge(early_rate, on=['route_id', 'full_hour'], how='left')
val_df   = val_df.merge(early_rate, on=['route_id', 'full_hour'], how='left')
test_df  = test_df.merge(early_rate, on=['route_id', 'full_hour'], how='left')

# מלא ערכים חסרים עם הממוצע הכללי
global_early_rate = (train_df['target'] == 'early').mean()
for df in [train_df, val_df, test_df]:
    df['route_hour_early_rate'] = df['route_hour_early_rate'].fillna(global_early_rate)

print(f"Global early rate: {global_early_rate:.3f}")
print(train_df['route_hour_early_rate'].describe().round(3))